<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 06. CLIP: 언어로 zero-shot 분류

## Learning Transferable Visual Models From Natural Language Supervision

- **원 논문:** [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020)
- **저자 / 발표:** Alec Radford et al. · ICML (2021)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** dual encoder, contrastive loss, zero-shot classification
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** image/text dual encoder와 symmetric contrastive loss를 학습해 retrieval·zero-shot 분류를 수행합니다.

**축소하거나 생략한 부분:** 4억 web pairs와 ResNet/ViT 대신 240개 합성 pair와 MLP/mean text encoder를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 1, step (1) | image/text dual encoder | L2 norm·similarity matrix |
| Figure 3 pseudocode | learned temperature와 대칭 CE | image↔text loss |
| Figure 1, step (2) | class text prototype | class×embedding shape |
| Figure 1, step (3), §3.1 | zero-shot prediction | class prediction accuracy |
| Figure 3 pseudocode 및 §2.2 | paired image↔text cosine retrieval | retrieval@1·similarity matrix |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L=\tfrac12\left[CE(S,\operatorname{diag})+CE(S^\top,\operatorname{diag})\right],
\quad S=\exp(t)I_eT_e^\top$$

- **기호 정의:** I_e/T_e는 정규화 image/text embedding, t는 logit scale입니다.
- **수식의 역할:** paired image-text를 같은 공간에 맞추는 대칭 dual-encoder 목적입니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 Figure 1, step (1)을 구현합니다.
- **입출력 shape:** image [B, 1, 8, 8], token [B, L] → similarity [B, B]
- **평가:** image↔text Recall@1과 class text prototype zero-shot accuracy
- **원문 대비 한계:** 4억 web pairs와 ResNet/ViT 대신 240개 합성 pair와 MLP/mean text encoder를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** Figure 1, step (1) / Figure 3 pseudocode
- **이 셀의 책임:** image/text dual encoder / learned temperature와 대칭 CE
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** L2 norm·similarity matrix / image↔text loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TinyCLIP(nn.Module):
    def __init__(self, vocab, dim=16):
        super().__init__()
        self.image = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, dim)
        )
        self.word = nn.Embedding(vocab, 24)
        self.text = nn.Linear(24, dim)
        self.logit_scale = nn.Parameter(
            torch.tensor(np.log(1 / 0.2), dtype=torch.float32)
        )

    def encode_image(self, x):
        return F.normalize(self.image(x), dim=1)

    def encode_text(self, t):
        return F.normalize(self.text(self.word(t).mean(1)), dim=1)


clip = TinyCLIP(int(tokens.max()) + 1)
assert torch.allclose(
    clip.encode_image(images[:8]).norm(dim=1), torch.ones(8), atol=1e-5
)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** Figure 3 pseudocode
- **이 셀의 책임:** learned temperature와 대칭 CE
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** image↔text loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def clip_loss(model, x, t):
    logits = (
        model.logit_scale.exp().clamp(max=100)
        * model.encode_image(x)
        @ model.encode_text(t).T
    )
    target = torch.arange(len(x), device=logits.device)
    return (
        F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)
    ) / 2, logits


loss, logits = clip_loss(clip, images[:24], tokens[:24])
assert logits.shape == (24, 24) and torch.isfinite(loss)

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** Figure 3 pseudocode
- **이 셀의 책임:** learned temperature와 대칭 CE
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** image↔text loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
clip = ACCELERATOR.move(clip)
optimizer = torch.optim.Adam(clip.parameters(), lr=0.012)
losses = []
x, t = ACCELERATOR.move(images[train_idx[:96]], tokens[train_idx[:96]])
for _ in range(140):
    loss, _ = clip_loss(clip, x, t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
assert losses[-1] < losses[0] * 0.7

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Figure 1, step (2) / Figure 1, step (3), §3.1 / Figure 3 pseudocode 및 §2.2
- **이 셀의 책임:** class text prototype / zero-shot prediction / paired image↔text cosine retrieval
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** class×embedding shape / class prediction accuracy / retrieval@1·similarity matrix. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
classes = class_ids.unique(sorted=True)
class_tokens = torch.stack([tokens[torch.where(class_ids == c)[0][0]] for c in classes])
with torch.no_grad():
    test_images, test_tokens, class_tokens_device, test_labels = ACCELERATOR.move(
        images[test_idx], tokens[test_idx], class_tokens, class_ids[test_idx]
    )
    zi = clip.encode_image(test_images)
    zt = clip.encode_text(test_tokens)
    retrieval = float(
        (zi @ zt.T)
        .argmax(1)
        .eq(torch.arange(len(test_idx), device=DEVICE))
        .float()
        .mean()
    )
    prototypes = clip.encode_text(class_tokens_device)
    prediction = (zi @ prototypes.T).argmax(1)
    zero_shot = float((prediction == test_labels).float().mean())
assert zero_shot > 1 / len(classes) and retrieval >= 1 / len(test_idx)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("CLIP loss")
ax[1].imshow((zi @ prototypes.T).cpu().numpy(), cmap="magma")
ax[1].set_title("image × class prompt")
plt.tight_layout()
plt.show()
print({"retrieval@1": round(retrieval, 3), "zero_shot_accuracy": round(zero_shot, 3)})

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** Figure 1, step (1) / Figure 3 pseudocode
- **이 셀의 책임:** image/text dual encoder / learned temperature와 대칭 CE
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** L2 norm·similarity matrix / image↔text loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    temperature: float = 0.15
    learning_rate: float = 0.02
    steps: int = 40


def prepare_views(
    image_batch: Tensor,
    text_batch: Tensor,
) -> tuple[Tensor, Tensor]:
    return image_batch, text_batch


class ClipPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        vocabulary = int(tokens.max()) + 1
        self.image_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, config.hidden_dim),
        )
        self.token_embedding = nn.Embedding(vocabulary, config.hidden_dim)
        self.text_projection = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.logit_scale = nn.Parameter(
            torch.log(torch.tensor(1.0 / config.temperature))
        )

    def encode_image(self, image_batch: Tensor) -> Tensor:
        return F.normalize(self.image_encoder(image_batch), dim=1)

    def encode_text(self, text_batch: Tensor) -> Tensor:
        text_hidden = self.token_embedding(text_batch).mean(dim=1)
        return F.normalize(self.text_projection(text_hidden), dim=1)

    def forward(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        image_embedding = self.encode_image(image_batch)
        text_embedding = self.encode_text(text_batch)
        scale = self.logit_scale.clamp(max=4.6052).exp()
        return scale * (image_embedding @ text_embedding.T)

    def compute_loss(self, similarity: Tensor) -> Tensor:
        target = torch.arange(len(similarity), device=similarity.device)
        return 0.5 * (
            F.cross_entropy(similarity, target) + F.cross_entropy(similarity.T, target)
        )

    def training_step(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        return self.compute_loss(self(image_batch, text_batch))

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** Figure 3 pseudocode
- **이 셀의 책임:** learned temperature와 대칭 CE
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** image↔text loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: ClipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    initial_logit_scale = model.logit_scale.detach().clone()
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(image_batch, text_batch)
        optimizer.zero_grad()
        loss.backward()
        assert model.logit_scale.grad is not None
        optimizer.step()
        with torch.no_grad():
            model.logit_scale.clamp_(max=4.6052)
        history.append(float(loss.detach().cpu()))
    assert not torch.allclose(initial_logit_scale, model.logit_scale.detach())
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** Figure 1, step (2) / Figure 1, step (3), §3.1 / Figure 3 pseudocode 및 §2.2
- **이 셀의 책임:** class text prototype / zero-shot prediction / paired image↔text cosine retrieval
- **Tensor shape 계약:** image/text [B,Dᵢ]/[B,T] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** class×embedding shape / class prediction accuracy / retrieval@1·similarity matrix. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: ClipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
    class_text_batch: Tensor,
    image_labels: Tensor,
    class_labels: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        image_embedding = model.encode_image(image_batch)
        text_embedding = model.encode_text(text_batch)
        similarity = image_embedding @ text_embedding.T
        target = torch.arange(len(similarity), device=similarity.device)
        recall = (similarity.argmax(1) == target).float().mean()
        class_prototypes = model.encode_text(class_text_batch)
        class_scores = image_embedding @ class_prototypes.T
        predicted_labels = class_labels[class_scores.argmax(dim=1)]
        zero_shot_accuracy = (predicted_labels == image_labels).float().mean()
    return {
        "heldout_image_to_text_recall1": float(recall.cpu()),
        "heldout_zero_shot_accuracy": float(zero_shot_accuracy.cpu()),
        "learned_temperature": float(model.logit_scale.exp().reciprocal().cpu()),
    }


portfolio_train_index = train_idx[:96]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_class_labels = class_ids[portfolio_train_index].unique(sorted=True)
portfolio_class_tokens = torch.stack(
    [
        tokens[
            portfolio_train_index[
                torch.where(class_ids[portfolio_train_index] == class_label)[0][0]
            ]
        ]
        for class_label in portfolio_class_labels
    ]
)
portfolio_train_images, portfolio_train_tokens = prepare_views(
    images[portfolio_train_index],
    tokens[portfolio_train_index],
)
(
    portfolio_train_images,
    portfolio_train_tokens,
    portfolio_test_images,
    portfolio_test_tokens,
    portfolio_class_tokens,
    portfolio_test_labels,
    portfolio_class_labels,
) = ACCELERATOR.move(
    portfolio_train_images,
    portfolio_train_tokens,
    images[portfolio_test_index],
    tokens[portfolio_test_index],
    portfolio_class_tokens,
    class_ids[portfolio_test_index],
    portfolio_class_labels,
)
portfolio_model = ACCELERATOR.move(ClipPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_images,
    portfolio_train_tokens,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_images,
    portfolio_test_tokens,
    portfolio_class_tokens,
    portfolio_test_labels,
    portfolio_class_labels,
)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["heldout_image_to_text_recall1"] >= 1 / len(test_idx)
assert portfolio_metrics["heldout_zero_shot_accuracy"] > 1 / len(portfolio_class_labels)
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 4억 web pairs와 ResNet/ViT 대신 240개 합성 pair와 MLP/mean text encoder를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.